# Seminar 2: Measuring Recommender Systems. Baselines for Recommendation

## Goals

In this seminar we will:
1. Learn about key **evaluation metrics** for recommender systems
2. Implement these metrics **from scratch**
3. Build two baseline recommendation algorithms: **Random** and **Top by Category**
4. Compare them using the implemented metrics on a real e-commerce dataset

## Why Measuring Matters

Building a recommender system is only half the battle — you need reliable
**offline evaluation metrics** to understand if your system actually works.
Different metrics capture different aspects of recommendation quality:

| Aspect | Metrics |
|--------|----------|
| Are the recommended items relevant? | Precision, Recall, Hit Rate |
| How well are relevant items ranked? | NDCG, MAP, MRR |
| Does the system cover the catalog? | Coverage |

In [ ]:
import os
import warnings
from collections import defaultdict

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("ggplot")

## 1. Dataset: Retailrocket E-commerce

We use the [Retailrocket](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset) dataset — a real-world e-commerce dataset containing:

- **~2.7M events** (view, add to cart, transaction) over 4.5 months
- **1.4M unique visitors** and **235K unique items**
- Each item belongs to a **category** — perfect for our "Top by Category" baseline

### Download

The dataset is hosted on Kaggle. You can download it using `opendatasets`:

```bash
pip install opendatasets
```

When prompted, enter your Kaggle username and API key (from https://www.kaggle.com/settings).

In [ ]:
!pip install opendatasets

In [ ]:
DATA_DIR = "ecommerce-dataset"

if not os.path.exists(DATA_DIR):
    import opendatasets as od
    od.download("https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset")

events = pl.read_csv(os.path.join(DATA_DIR, "events.csv"))
print(f"Events shape: {events.shape}")
events.head()

In [ ]:
item_props = pl.concat([
    pl.read_csv(os.path.join(DATA_DIR, "item_properties_part1.csv")),
    pl.read_csv(os.path.join(DATA_DIR, "item_properties_part2.csv")),
])

item_categories = (
    item_props
    .filter(pl.col("property") == "categoryid")
    .select(["itemid", pl.col("value").alias("categoryid")])
    .unique(subset=["itemid"], keep="last")
)

print(f"Items with categories: {len(item_categories)}")
print(f"Unique categories: {item_categories['categoryid'].n_unique()}")

events = events.join(item_categories, on="itemid", how="inner")
print(f"Events with categories: {events.shape}")
events.head()

## 2. Exploratory Data Analysis

In [ ]:
print(f"Number of events:     {len(events):,}")
print(f"Unique users:         {events['visitorid'].n_unique():,}")
print(f"Unique items:         {events['itemid'].n_unique():,}")
print(f"Unique categories:    {events['categoryid'].n_unique():,}")
print(f"Event types:          {events['event'].unique().to_list()}")
print()

event_counts = (
    events.group_by("event")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
)
print("Event type distribution:")
print(event_counts)

sparsity = 1 - len(events) / (events['visitorid'].n_unique() * events['itemid'].n_unique())
print(f"\nInteraction matrix sparsity: {sparsity:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 10))

user_event_counts = events.group_by("visitorid").agg(pl.count().alias("n_events")) # visitorid == userid
axes[0].hist(user_event_counts["n_events"].to_numpy(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Number of events")
axes[0].set_ylabel("Number of users")
axes[0].set_title("Events per User")
axes[0].set_yscale("log")

item_event_counts = events.group_by("itemid").agg(pl.count().alias("n_events"))
axes[1].hist(item_event_counts["n_events"].to_numpy(), bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Number of events")
axes[1].set_ylabel("Number of items")
axes[1].set_title("Events per Item")
axes[1].set_yscale("log")

cat_counts = (
    events.group_by("categoryid")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
    .head(15)
)
axes[2].barh(
    cat_counts["categoryid"].cast(pl.Utf8).to_list()[::-1],
    cat_counts["count"].to_list()[::-1],
    alpha=0.7,
)
axes[2].set_xlabel("Number of events")
axes[2].set_title("Top 15 Categories")

plt.tight_layout()
plt.show()

## 3. Data Preparation

### Filtering Active Users

For meaningful evaluation, we keep only users with at least **5 interactions**. Users with fewer interactions don't provide enough signal for personalized recommendations.

### Train/Test Split: Leave-Last-Out

We use the **leave-last-out** strategy:
- For each user, the **last interaction** (by timestamp) goes to the **test set**
- All previous interactions form the **training set**

This mimics a real-world scenario: we train on historical data and evaluate on the most recent behavior.

In [ ]:
MIN_INTERACTIONS = 5

user_counts = events.group_by("visitorid").agg(pl.count().alias("n_events"))
active_users = user_counts.filter(pl.col("n_events") >= MIN_INTERACTIONS)["visitorid"]
data = events.filter(pl.col("visitorid").is_in(active_users))

print(f"Users after filtering (>= {MIN_INTERACTIONS} events): {data['visitorid'].n_unique():,}")
print(f"Events after filtering: {len(data):,}")
print(f"Items after filtering: {data['itemid'].n_unique():,}")

In [ ]:
data = data.sort(["visitorid", "timestamp"])

data_ranked = data.with_columns(
    pl.col("timestamp")
    .rank("ordinal", descending=True)
    .over("visitorid")
    .alias("time_rank")
)

test = data_ranked.filter(pl.col("time_rank") == 1).drop("time_rank")
train = data_ranked.filter(pl.col("time_rank") > 1).drop("time_rank")

print(f"Train set: {len(train):,} events")
print(f"Test set:  {len(test):,} events (1 per user)")
print(f"Users in test: {test['visitorid'].n_unique():,}")

In [ ]:
train_user_items = {}
for row in train.group_by("visitorid").agg(pl.col("itemid")).iter_rows():
    train_user_items[row[0]] = set(row[1])

ground_truth = {}
for row in test.select(["visitorid", "itemid"]).iter_rows():
    ground_truth[row[0]] = {row[1]}

all_items = set(data["itemid"].unique().to_list())

K = 10
user_ids = list(ground_truth.keys())

print(f"Users for evaluation: {len(user_ids):,}")
print(f"Catalog size: {len(all_items):,}")

## 4. Recommendation Quality Metrics

We implement 7 key metrics from scratch. Each metric captures a different aspect of recommendation quality.

All our metrics work with:
- `recommendations: dict[user_id, list[item_id]]` — ordered list of K recommended items per user
- `ground_truth: dict[user_id, set[item_id]]` — set of relevant (ground truth) items per user

> **Note:** With leave-last-out evaluation, each user has exactly 1 relevant item. This means Hit Rate = Recall and MAP = MRR. We implement all metrics separately since they differ in the general case (multiple relevant items per user).

### 4.1 Hit Rate @ K

The simplest metric: what **fraction of users** got at least one relevant item in their top-K recommendations?

$$\text{HitRate@K} = \frac{1}{|U|} \sum_{u \in U} \mathbb{1}\left[\text{Rel}(u) \cap \text{Rec}_K(u) \neq \emptyset\right]$$

- Values range from 0 to 1
- Easy to interpret: "for X% of users we made at least one good recommendation"

In [ ]:
def hit_rate_at_k(recommendations, ground_truth, k):
    hits = 0
    total = 0
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        if set(rec_list[:k]) & ground_truth[user_id]:
            hits += 1
        total += 1
    return hits / total if total > 0 else 0.0

### 4.2 Precision @ K

What **fraction of recommended items** are actually relevant?

$$\text{Precision@K} = \frac{1}{|U|} \sum_{u \in U} \frac{|\text{Rel}(u) \cap \text{Rec}_K(u)|}{K}$$

- With leave-last-out (1 relevant item), maximum possible Precision@10 = 0.1

In [ ]:
def precision_at_k(recommendations, ground_truth, k):
    precisions = []
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        relevant = len(set(rec_list[:k]) & ground_truth[user_id])
        precisions.append(relevant / k)
    return np.mean(precisions) if precisions else 0.0

### 4.3 Recall @ K

What **fraction of relevant items** did we manage to recommend?

$$\text{Recall@K} = \frac{1}{|U|} \sum_{u \in U} \frac{|\text{Rel}(u) \cap \text{Rec}_K(u)|}{|\text{Rel}(u)|}$$

- With leave-last-out (1 relevant item), Recall@K equals Hit Rate@K

In [ ]:
def recall_at_k(recommendations, ground_truth, k):
    recalls = []
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        relevant_found = len(set(rec_list[:k]) & ground_truth[user_id])
        total_relevant = len(ground_truth[user_id])
        recalls.append(relevant_found / total_relevant if total_relevant > 0 else 0.0)
    return np.mean(recalls) if recalls else 0.0

### 4.4 Mean Average Precision @ K (MAP@K)

MAP accounts for the **position** of relevant items. A relevant item ranked 1st is better than one ranked 10th.

$$\text{AP@K}(u) = \frac{1}{\min(|\text{Rel}(u)|,\, K)} \sum_{i=1}^{K} P(i) \cdot \text{rel}(i)$$

$$\text{MAP@K} = \frac{1}{|U|} \sum_{u \in U} \text{AP@K}(u)$$

where $P(i)$ is precision at position $i$ and $\text{rel}(i)$ is 1 if the item at position $i$ is relevant.

In [ ]:
def ap_at_k(rec_list, relevant, k):
    """Average Precision @ K for a single user."""
    hits = 0
    sum_precisions = 0.0
    for i, item in enumerate(rec_list[:k]):
        if item in relevant:
            hits += 1
            sum_precisions += hits / (i + 1)
    return sum_precisions / min(len(relevant), k) if relevant else 0.0


def map_at_k(recommendations, ground_truth, k):
    aps = []
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        aps.append(ap_at_k(rec_list, ground_truth[user_id], k))
    return np.mean(aps) if aps else 0.0

### 4.5 Mean Reciprocal Rank (MRR)

How **high** is the first relevant item ranked?

$$\text{MRR@K} = \frac{1}{|U|} \sum_{u \in U} \frac{1}{\text{rank}_u}$$

where $\text{rank}_u$ is the position of the first relevant item in the recommendation list (0 if not found in top-K).

- MRR = 1 means every user's relevant item is ranked first
- With leave-last-out (1 relevant item), MRR@K = MAP@K

In [ ]:
def mrr_at_k(recommendations, ground_truth, k):
    reciprocal_ranks = []
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        rr = 0.0
        for i, item in enumerate(rec_list[:k]):
            if item in ground_truth[user_id]:
                rr = 1.0 / (i + 1)
                break
        reciprocal_ranks.append(rr)
    return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0

### 4.6 Normalized Discounted Cumulative Gain @ K (NDCG@K)

NDCG uses **logarithmic discounting** — items at lower positions contribute less to the score.

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{\text{rel}(i)}{\log_2(i + 1)}$$

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

where IDCG@K is the ideal DCG (all relevant items placed at the top positions).

In [ ]:
def dcg_at_k(rec_list, relevant, k):
    dcg = 0.0
    for i, item in enumerate(rec_list[:k]):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    return dcg


def ndcg_at_k(recommendations, ground_truth, k):
    ndcgs = []
    for user_id, rec_list in recommendations.items():
        if user_id not in ground_truth:
            continue
        dcg = dcg_at_k(rec_list, ground_truth[user_id], k)
        ideal_hits = min(len(ground_truth[user_id]), k)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs) if ndcgs else 0.0

### 4.7 Coverage

What **fraction of the item catalog** appears in at least one user's recommendations?

$$\text{Coverage@K} = \frac{\left|\bigcup_{u \in U} \text{Rec}_K(u)\right|}{|\text{Items}|}$$

- Low coverage = the system only recommends popular items (filter bubble)
- High coverage = the system explores the full catalog

In [ ]:
def coverage_at_k(recommendations, all_items, k):
    recommended = set()
    for rec_list in recommendations.values():
        recommended.update(rec_list[:k])
    return len(recommended) / len(all_items) if all_items else 0.0

### Verification on Toy Data

Let's verify our implementations on a small example where we can compute results by hand.

In [ ]:
toy_recs = {
    "user_A": [1, 2, 3, 4, 5],
    "user_B": [6, 7, 8, 9, 10],
    "user_C": [11, 12, 13, 14, 15],
}
toy_gt = {
    "user_A": {3},      # relevant at position 3
    "user_B": {6},      # relevant at position 1
    "user_C": {20},     # relevant NOT in recommendations
}
toy_items = set(range(1, 21))

print("=== Toy Example Verification (K=5) ===")
print(f"Hit Rate@5:  {hit_rate_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.6667 = 2/3 users hit)")
print(f"Precision@5: {precision_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.1333 = mean(1/5, 1/5, 0))")
print(f"Recall@5:    {recall_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.6667 = same as hit rate)")
print(f"MAP@5:       {map_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.4444 = mean(1/3, 1, 0))")
print(f"MRR@5:       {mrr_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.4444 = same as MAP)")
print(f"NDCG@5:      {ndcg_at_k(toy_recs, toy_gt, 5):.4f}  (expected: 0.5000)")
print(f"Coverage@5:  {coverage_at_k(toy_recs, toy_items, 5):.4f}  (expected: 0.7500 = 15/20)")

In [ ]:
def evaluate_all(recommendations, ground_truth, all_items, k):
    return {
        "HitRate@K": hit_rate_at_k(recommendations, ground_truth, k),
        "Precision@K": precision_at_k(recommendations, ground_truth, k),
        "Recall@K": recall_at_k(recommendations, ground_truth, k),
        "MAP@K": map_at_k(recommendations, ground_truth, k),
        "MRR@K": mrr_at_k(recommendations, ground_truth, k),
        "NDCG@K": ndcg_at_k(recommendations, ground_truth, k),
        "Coverage@K": coverage_at_k(recommendations, all_items, k),
    }

## 5. Recommendation Algorithms

We implement two baselines:

1. **Random** — randomly sample K items (excluding seen). The simplest possible baseline.
2. **Top by Category** — recommend the most popular items from the user's preferred categories. A simple but effective content-based approach.

### 5.1 Random Recommendations

For each user, we randomly select K items from the catalog, excluding items the user has already interacted with. This establishes a **lower bound** — any reasonable algorithm should beat random.

In [ ]:
def recommend_random(user_ids, train_user_items, all_items, k=10, seed=42):
    rng = np.random.default_rng(seed)
    all_items_list = list(all_items)
    n_items = len(all_items_list)
    recommendations = {}

    for user_id in user_ids:
        seen = train_user_items.get(user_id, set())
        rec = []
        rec_set = set()
        while len(rec) < k:
            idx = rng.integers(0, n_items)
            item = all_items_list[idx]
            if item not in seen and item not in rec_set:
                rec.append(item)
                rec_set.add(item)
        recommendations[user_id] = rec

    return recommendations

### 5.2 Top by Category

For each user:
1. Count how many times the user interacted with items in each category (from training data)
2. Rank categories by user's interaction count (most interacted = most preferred)
3. From preferred categories, recommend the **most globally popular items** the user hasn't seen yet

This baseline uses both **user preferences** (category affinity) and **item popularity** (within category).

In [ ]:
# Prepare: user -> sorted list of preferred category IDs
user_cat_prefs = {}
for row in (
    train.select(["visitorid", "categoryid"])
    .group_by(["visitorid", "categoryid"])
    .agg(pl.count().alias("cnt"))
    .sort(["visitorid", "cnt"], descending=[False, True])
    .iter_rows(named=True)
):
    uid = row["visitorid"]
    cat = row["categoryid"]
    user_cat_prefs.setdefault(uid, []).append(cat)

# Prepare: category -> list of item IDs sorted by global popularity (descending)
category_top_items = {}
for row in (
    train.select(["categoryid", "itemid"])
    .group_by(["categoryid", "itemid"])
    .agg(pl.count().alias("popularity"))
    .sort(["categoryid", "popularity"], descending=[False, True])
    .iter_rows(named=True)
):
    cat = row["categoryid"]
    category_top_items.setdefault(cat, []).append(row["itemid"])

# Prepare: global popularity fallback
global_top_items = (
    train.group_by("itemid")
    .agg(pl.count().alias("popularity"))
    .sort("popularity", descending=True)
    ["itemid"].to_list()
)

print(f"Users with category preferences: {len(user_cat_prefs):,}")
print(f"Categories with ranked items: {len(category_top_items):,}")

In [ ]:
def recommend_top_by_category(user_ids, train_user_items, user_cat_prefs,
                              category_top_items, global_top_items, k=10):
    recommendations = {}

    for user_id in user_ids:
        seen = train_user_items.get(user_id, set())
        cats = user_cat_prefs.get(user_id, [])
        rec = []
        rec_set = set()

        for cat_id in cats:
            for item_id in category_top_items.get(cat_id, []):
                if item_id not in seen and item_id not in rec_set:
                    rec.append(item_id)
                    rec_set.add(item_id)
                if len(rec) >= k:
                    break
            if len(rec) >= k:
                break

        # Fallback: fill remaining slots with globally popular items
        if len(rec) < k:
            for item_id in global_top_items:
                if item_id not in seen and item_id not in rec_set:
                    rec.append(item_id)
                    rec_set.add(item_id)
                if len(rec) >= k:
                    break

        recommendations[user_id] = rec[:k]

    return recommendations

## 6. Generate Recommendations and Evaluate

In [ ]:
print("Generating Random recommendations...")
recs_random = recommend_random(user_ids, train_user_items, all_items, k=K)

print("Generating Top-by-Category recommendations...")
recs_top_cat = recommend_top_by_category(
    user_ids, train_user_items, user_cat_prefs,
    category_top_items, global_top_items, k=K
)

print(f"Random: generated for {len(recs_random):,} users")
print(f"Top-by-Category: generated for {len(recs_top_cat):,} users")

In [ ]:
print(f"Evaluating with K={K}...\n")

results_random = evaluate_all(recs_random, ground_truth, all_items, K)
results_top_cat = evaluate_all(recs_top_cat, ground_truth, all_items, K)

results_df = pl.DataFrame({
    "Metric": list(results_random.keys()),
    "Random": [round(v, 6) for v in results_random.values()],
    "TopByCategory": [round(v, 6) for v in results_top_cat.values()],
})

print(results_df)

In [ ]:
metrics = list(results_random.keys())
random_vals = list(results_random.values())
top_cat_vals = list(results_top_cat.values())

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width / 2, random_vals, width, label="Random", alpha=0.8)
bars2 = ax.bar(x + width / 2, top_cat_vals, width, label="Top by Category", alpha=0.8)

ax.set_ylabel("Score")
ax.set_title(f"Algorithm Comparison (K={K})")
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=30, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 7. Conclusion

### Key Observations

1. **Top by Category significantly outperforms Random** on all ranking metrics (Hit Rate, Precision, Recall, MAP, MRR, NDCG). This confirms that even a simple content-based signal (category preference) is far more valuable than no signal at all.

2. **Random has higher Coverage** — it recommends items uniformly across the catalog, while Top by Category focuses on popular items in preferred categories. This illustrates the classic **relevance vs. diversity trade-off**.

3. **MAP@K = MRR@K** in our evaluation because we used leave-last-out (1 relevant item per user). Similarly, **Hit Rate@K = Recall@K**. In practice, with multiple relevant items per user, these metrics diverge.

### Limitations

- **Random** ignores all user preferences — it's only useful as a lower bound.
- **Top by Category** relies on popularity within categories, which leads to a popularity bias. It also requires item category metadata.
- Both methods are **non-personalized** beyond category preferences — they don't capture individual user taste within a category.